# Agent Sandbox
Breaks down `agent.py` step by step so each piece can be inspected and tested independently.

In [23]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv('../.env')

True

## 1. Imports

In [24]:
import aiosqlite
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, MessagesState, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
from langfuse.langchain import CallbackHandler
from deps import make_obj
from tool import document_search

## 2. Prompt
The system prompt tells the LLM who it is and what it can answer.

In [25]:
AGENT_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are DiabeticAssist, a clinical reference chatbot specializing in diabetes care.
You answer questions using verified medical documents only.
Always cite the source document when referencing specific facts.
Never provide personal medical advice or recommend starting/stopping treatments.
Add a safety disclaimer when discussing dosages, insulin regimens, or drug interactions.
Focus on topics including: Type 1 and Type 2 diabetes, prediabetes, gestational diabetes,
blood glucose management, HbA1c targets, medications (metformin, insulin, GLP-1 agonists, SGLT2 inhibitors),
dietary guidance, and diabetes complications.
If a question is outside the scope of available documents, say so clearly."""),
    MessagesPlaceholder("messages"),
])

print(AGENT_PROMPT)

input_variables=['messages'] input_types={'messages': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_core.messages.function.Funct

3. LLM + Tools
`bind_tools` tells the LLM it can call `document_search` when it needs to retrieve context.

In [26]:
tools = [document_search]
tool_node = ToolNode(tools)
llm = AGENT_PROMPT | make_obj().bind_tools(tools)

print("Tools:", [t.name for t in tools])

Tools: ['document_search']


build tool call graph

In [27]:
#build tool call graph
def call_model(state: MessagesState):
    response = llm.invoke({"messages": state["messages"]})
    return {"messages": [response]}

In [28]:
tool_call=llm.invoke({"messages": [HumanMessage(content="What is the HbA1c target for type 2 diabetes?")]})
#print tool calls paramters nice and readable format

print(tool_call.tool_calls[0]['name'])
print(tool_call.tool_calls[0][ 'args'])

document_search
{'query': 'HbA1c target for type 2 diabetes guideline HbA1c target type 2 ADA NICE SIGN IDF consensus', 'k': 5}


In [18]:
message=llm.invoke({"messages": [HumanMessage(content="what is blue?")]})
print(message)

content='That question is outside the scope of my diabetes-focused, medical-document-based knowledge. I can’t provide a verified, document-cited answer about general topics like the color “blue.”\n\nDo you mean:\n- the color blue (non-medical), \n- “feeling blue” (sadness), or \n- a medical sign such as cyanosis or “blue” toes/fingers (which can be relevant to vascular or diabetic complications)?\n\nIf you mean a medical sign (cyanosis, blue toes, etc.), tell me which and I’ll provide medically referenced information and citations from verified sources. If you mean the color or an emotional state, I can give a general explanation but it won’t be based on the medical documents I use for diabetes topics.' additional_kwargs={} response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-5-mini-2025-08-07', 'service_tier': 'priority', 'model_provider': 'openai'} id='lc_run--019e1018-303c-7c03-8bc3-2f9ee1b04c52' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 329, 'out

## 4. Graph Nodes
`call_model` runs the LLM. `should_continue` decides whether to call a tool or stop.

In [ ]:
def call_model(state: MessagesState):
    response = llm.invoke({"messages": state["messages"]})
    return {"messages": [response]}

def should_continue(state: MessagesState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"   # LLM wants to call a tool
    return END            # LLM produced a final answer

## 5. Build the Graph
Wires agent → tools → agent in a loop until `should_continue` returns END.

In [ ]:
def build_graph():
    graph = StateGraph(MessagesState)
    graph.add_node("agent", call_model)
    graph.add_node("tools", tool_node)
    graph.set_entry_point("agent")
    graph.add_conditional_edges("agent", should_continue)
    graph.add_edge("tools", "agent")
    return graph

print("Graph built")

## 6. Compile with Checkpointer
The checkpointer saves message history to SQLite so the agent remembers previous turns.

In a notebook we can `await` directly — no need to wrap in `async def`.

In [ ]:
conn = await aiosqlite.connect("sandbox.db")
checkpointer = AsyncSqliteSaver(conn)
await checkpointer.setup()

agent = build_graph().compile(checkpointer=checkpointer)
print("Agent compiled")

## 7. Run the Agent
Send a message and collect the full response.

In [ ]:
langfuse_handler = CallbackHandler()
config = {"configurable": {"thread_id": "sandbox-1"}, "callbacks": [langfuse_handler]}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="What is the HbA1c target for type 2 diabetes?")]},
    config=config,
)

print(response["messages"][-1].content)